In [4]:
# qiskit_qml_diagnostic_and_fix.py
# Try multiple quantum-compatible embeddings/kernels and compare to classical baseline.
# Compatible with Qiskit ~1.4.4

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# qiskit
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler

# -----------------------
# Config
# -----------------------
FILE_NAME = "Heart_data.csv"
TEST_SIZE = 0.2
RANDOM_SEED = 24
SCALE_TO_PI = True
FEATURE_MAP_REPS = 1
PCA_FOR_PROBS = 8        # reduce the 64-d probability vector to this (tunable)
PCA_FOR_STATE = 8        # reduce real+imag statevector dims to this (tunable)
INCLUDE_PAIRWISE_ZZ = True
VERBOSE = True

# -----------------------
# Load & normalize
# -----------------------
df = pd.read_csv(FILE_NAME)
FEATURE_COLUMNS = ['Age','Gender','BloodPressure','Cholesterol','HeartRate','QuantumPatternFeature']
X_raw = df[FEATURE_COLUMNS].astype(float).values
y = df['HeartDisease'].astype(int).values

scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X_raw)
if SCALE_TO_PI:
    X_for_q = X_norm * np.pi
else:
    X_for_q = X_norm.copy()

n_samples, n_features = X_for_q.shape
print(f"Loaded {FILE_NAME}: samples={n_samples}, features={n_features}")
unique, counts = np.unique(y, return_counts=True)
print("Label distribution:", dict(zip(unique, counts)))

# -----------------------
# Build encoding circuits
# -----------------------
feature_map = ZZFeatureMap(feature_dimension=n_features, reps=FEATURE_MAP_REPS)

def build_encoding_circuit(x):
    params = {}
    try:
        for i,p in enumerate(feature_map.parameters):
            params[p] = float(x[i])
        circ = feature_map.assign_parameters(params)
    except Exception:
        # fallback simple RX encoding
        from qiskit import QuantumCircuit
        qc = QuantumCircuit(n_features)
        for i,val in enumerate(x):
            qc.rx(float(val), i)
        circ = qc
    circ.measure_all()
    return circ

circuits = [build_encoding_circuit(x) for x in X_for_q]

# -----------------------
# Run StatevectorSampler once for all circuits (get statevectors)
# -----------------------
sampler = StatevectorSampler()
job = sampler.run(circuits)
res = job.result()

def extract_statevector(res_entry):
    # robust extraction across Qiskit versions
    try:
        sv = getattr(res_entry.data, "statevector", None)
        if sv is not None:
            return np.asarray(sv, dtype=np.complex128)
    except Exception:
        pass
    try:
        sv = getattr(res_entry.data, "state", None)
        if sv is not None:
            return np.asarray(sv, dtype=np.complex128)
    except Exception:
        pass
    try:
        sv = getattr(res_entry, "statevector", None)
        if sv is not None:
            return np.asarray(sv, dtype=np.complex128)
    except Exception:
        pass
    try:
        d = res_entry.data
        if isinstance(d, dict):
            for k in ("statevector","state"):
                if k in d:
                    return np.asarray(d[k], dtype=np.complex128)
    except Exception:
        pass
    # last resort
    return None

statevectors = []
for i in range(len(circuits)):
    try:
        res_entry = res[i]
    except Exception:
        try:
            res_entry = res._result_list[i]
        except Exception:
            res_entry = res
    sv = extract_statevector(res_entry)
    if sv is None:
        # fallback: zero state (shouldn't happen)
        m = 2**n_features
        z = np.zeros(m, dtype=np.complex128)
        z[0] = 1.0
        sv = z
    # normalize (safety)
    sv = np.asarray(sv, dtype=np.complex128)
    norm = np.linalg.norm(sv)
    if norm == 0 or not np.isfinite(norm):
        sv = np.zeros_like(sv)
        sv[0] = 1.0
    else:
        sv = sv / norm
    statevectors.append(sv)
statevectors = np.vstack(statevectors)  # shape (samples, 2^n)
print("Extracted statevectors shape:", statevectors.shape)

# -----------------------
# Helper: compute classical embeddings
# A) Z & pairwise ZZ
# B) full computational-basis probabilities (then PCA)
# C) real+imag of statevector (then PCA)
# D) quantum kernel (fidelity) for SVM
# -----------------------
def embedding_Z_ZZ_from_statevector(sv, include_pairwise=True):
    probs = np.abs(sv)**2
    nstates = len(probs)
    nqubits = int(np.round(np.log2(nstates)))
    bits = np.array([list(map(int, format(i, "0{}b".format(nqubits)))) for i in range(nstates)], dtype=int)
    signs = 1 - 2*bits  # 0->1,1->-1
    Z = np.sum(probs[:,None] * signs, axis=0)
    if include_pairwise:
        ZZ = []
        for i in range(nqubits):
            for j in range(i+1, nqubits):
                zz_vals = signs[:,i] * signs[:,j]
                zz_val = np.sum(probs * zz_vals)
                ZZ.append(zz_val)
        return np.concatenate([Z, np.array(ZZ)])
    else:
        return Z

# compute embeddings for every sample
emb_ZZZ = np.vstack([embedding_Z_ZZ_from_statevector(sv, include_pairwise=INCLUDE_PAIRWISE_ZZ) for sv in statevectors])
print("Z/ZZ embedding shape:", emb_ZZZ.shape)

# probabilities
probs_all = np.abs(statevectors)**2
print("Prob vectors shape:", probs_all.shape)

# real+imag flatten
realimag = np.hstack([statevectors.real, statevectors.imag])
print("Real+Imag embedding raw shape:", realimag.shape)

# -----------------------
# PCA reductions for high-d embeddings
# -----------------------
pca_probs = PCA(n_components=min(PCA_FOR_PROBS, probs_all.shape[1]), random_state=RANDOM_SEED)
probs_pca = pca_probs.fit_transform(probs_all)

pca_state = PCA(n_components=min(PCA_FOR_STATE, realimag.shape[1]), random_state=RANDOM_SEED)
state_pca = pca_state.fit_transform(realimag)

print("Prob-PCA shape:", probs_pca.shape, "State-PCA shape:", state_pca.shape)

# -----------------------
# Prepare train/test splits per embedding
# -----------------------
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_norm, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
X_train_z, X_test_z, _, _ = train_test_split(emb_ZZZ, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
X_train_ppca, X_test_ppca, _, _ = train_test_split(probs_pca, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
X_train_spca, X_test_spca, _, _ = train_test_split(state_pca, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

# -----------------------
# Classical baselines on each embedding
# -----------------------
def run_models(Xtr, Xte, ytr, yte, label):
    models = {
        "Logistic": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED),
        "SVM": SVC(kernel='rbf', gamma='scale', random_state=RANDOM_SEED)
    }
    results = {}
    for name,m in models.items():
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)
        acc = accuracy_score(yte, pred)
        results[name] = (acc, classification_report(yte, pred, output_dict=True), confusion_matrix(yte, pred))
        print(f"{label} - {name} accuracy: {acc*100:.2f}%")
    return results

print("\n--- Baselines on raw features ---")
res_raw = run_models(X_train_raw, X_test_raw, y_train, y_test, "Raw")

print("\n--- Baselines on Z/ZZ embedding ---")
res_z = run_models(X_train_z, X_test_z, y_train, y_test, "Z/ZZ")

print("\n--- Baselines on Prob-PCA embedding ---")
res_ppca = run_models(X_train_ppca, X_test_ppca, y_train, y_test, "Prob-PCA")

print("\n--- Baselines on State Real+Imag PCA embedding ---")
res_spca = run_models(X_train_spca, X_test_spca, y_train, y_test, "State-PCA")

# -----------------------
# Quantum kernel (fidelity) SVM
# Build kernel matrices:
# K_train[i,j] = |<psi_i | psi_j>|^2
# K_test[i,j] = |<psi_test_i | psi_train_j>|^2
# -----------------------
print("\n--- Quantum kernel (fidelity) SVM ---")
# build train and test arrays in same order as X_train_raw indices
# Need to get indices used in train_test_split; easiest is to split indices once and reuse
indices = np.arange(len(X_norm))
itr, ite, ytr_idx, yte_idx = train_test_split(indices, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
# statevectors[itr] are training states, statevectors[ite] test states
sv_train = statevectors[itr]
sv_test = statevectors[ite]

# compute gram matrices using complex overlaps
def fidelity_matrix(A, B):
    # A: m x d, B: n x d
    # returns matrix M (m x n) with |<A_i|B_j>|^2
    M = np.abs(np.dot(np.conjugate(A), B.T))**2
    return M

K_train = fidelity_matrix(sv_train, sv_train)
K_test = fidelity_matrix(sv_test, sv_train)

# Train SVM with precomputed kernel
svc = SVC(kernel='precomputed', probability=False, random_state=RANDOM_SEED)
svc.fit(K_train, ytr_idx)
pred_test = svc.predict(K_test)
acc_k = accuracy_score(yte_idx, pred_test)
print(f"Quantum-kernel SVM accuracy: {acc_k*100:.2f}%")
print("Classification report (quantum kernel):")
print(classification_report(yte_idx, pred_test))
print("Confusion matrix (quantum kernel):\n", confusion_matrix(yte_idx, pred_test))

# -----------------------
# Summarize best results
# -----------------------
summary = {
    "Raw Logistic": res_raw["Logistic"][0],
    "Z/ZZ Logistic": res_z["Logistic"][0],
    "Prob-PCA Logistic": res_ppca["Logistic"][0],
    "State-PCA Logistic": res_spca["Logistic"][0],
    "Quantum-kernel SVM": acc_k
}
print("\nSummary accuracies:")
for k,v in summary.items():
    print(f"{k}: {v*100:.2f}%")

# If quantum-kernel SVM matches raw logistic (~92%), great. If not, we go next step.


Loaded Heart_data.csv: samples=500, features=6
Label distribution: {np.int64(0): np.int64(200), np.int64(1): np.int64(300)}
Extracted statevectors shape: (500, 64)
Z/ZZ embedding shape: (500, 21)
Prob vectors shape: (500, 64)
Real+Imag embedding raw shape: (500, 128)


/learning/ibm/venv/lib/python3.11/site-packages/sklearn/decomposition/_pca.py:646: RuntimeWarning: invalid value encountered in divide
  explained_variance_ratio_ = explained_variance_ / total_var
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/decomposition/_pca.py:646: RuntimeWarning: invalid value encountered in divide
  explained_variance_ratio_ = explained_variance_ / total_var


Prob-PCA shape: (500, 8) State-PCA shape: (500, 8)

--- Baselines on raw features ---
Raw - Logistic accuracy: 91.00%
Raw - RandomForest accuracy: 91.00%
Raw - SVM accuracy: 87.00%

--- Baselines on Z/ZZ embedding ---
Z/ZZ - Logistic accuracy: 60.00%


/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm

Z/ZZ - RandomForest accuracy: 60.00%
Z/ZZ - SVM accuracy: 60.00%

--- Baselines on Prob-PCA embedding ---
Prob-PCA - Logistic accuracy: 60.00%


/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm

Prob-PCA - RandomForest accuracy: 60.00%
Prob-PCA - SVM accuracy: 60.00%

--- Baselines on State Real+Imag PCA embedding ---
State-PCA - Logistic accuracy: 60.00%
State-PCA - RandomForest accuracy: 60.00%
State-PCA - SVM accuracy: 60.00%

--- Quantum kernel (fidelity) SVM ---
Quantum-kernel SVM accuracy: 60.00%
Classification report (quantum kernel):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        40
           1       0.60      1.00      0.75        60

    accuracy                           0.60       100
   macro avg       0.30      0.50      0.38       100
weighted avg       0.36      0.60      0.45       100

Confusion matrix (quantum kernel):
 [[ 0 40]
 [ 0 60]]

Summary accuracies:
Raw Logistic: 91.00%
Z/ZZ Logistic: 60.00%
Prob-PCA Logistic: 60.00%
State-PCA Logistic: 60.00%
Quantum-kernel SVM: 60.00%


/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/learning/ibm